# EXP004 — LLM Enrichment Signal Value

**Question:** Does the `employer_desc` field generated by Claude API during node enrichment improve downstream query ranking quality?
**Hypothesis:** employer_desc adds modest signal (~2–5% NDCG) but rank_correlation > 0.90, suggesting marginal benefit relative to API cost.
**Success criterion:** with_desc NDCG@10 ≥ no_desc NDCG@10 + 0.02 AND rank_correlation < 0.95.
**Production impact:** `netweave/src/enrich.py` → whether employer_desc is included in embedding input.
**Author:** Chuck Wiley  **Date:** 2026-06-23

In [ ]:
import sys
sys.path.insert(0, ".")

import mlflow
import mlflow.tracking
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import anthropic
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from lib.niw_graph import load_graph
from lib.niw_metrics import ndcg_at_k, build_ground_truth, rank_correlation
from lib.niw_mlflow import start_run, log_result, compare_runs

mlflow.set_tracking_uri("sqlite:///mlflow.db")

EXPERIMENT_ID = "EXP004"
EXPERIMENT_NAME = "LLM Enrichment Signal Value"
mlflow.set_experiment(EXPERIMENT_NAME)

client = anthropic.Anthropic()

In [ ]:
# Always load from shared/ — never modify source data
G = load_graph(
    nodes_path="shared/nodes.parquet",
    edges_path="shared/edges.parquet"
)
print(f"Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

encoder = SentenceTransformer("all-MiniLM-L6-v2")

GOAL_QUERY = "Find MASLD investors and clinical partners"
GOAL_TAGS = ["medical_diagnostics", "biotech", "investor"]
relevant_set = build_ground_truth(G, GOAL_TAGS)

In [ ]:
def get_employer_desc(node_data: dict) -> str:
    """Generate employer description via Claude API."""
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=80,
        messages=[{
            "role": "user",
            "content": (
                f"In one sentence, describe what {node_data.get('employer', 'this company')} does "
                f"in the context of healthcare and life sciences. Be specific about their domain."
            )
        }],
    )
    return response.content[0].text.strip()

def embed_nodes(G: nx.DiGraph, include_desc: bool, descs: dict) -> tuple:
    """Embed nodes into vector space. Returns (node_ids, embeddings)."""
    node_ids = list(G.nodes())
    texts = []
    for n in node_ids:
        d = G.nodes[n]
        text = f"{d.get('employer', '')} {d.get('position', '')} {' '.join(d.get('domain_tags', []))}"
        if include_desc and n in descs:
            text += f" {descs[n]}"
        texts.append(text)
    embs = encoder.encode(texts).astype("float32")
    return node_ids, embs

variants = {
    "no_desc":   {"include_desc": False},
    "with_desc": {"include_desc": True},
}

In [ ]:
# Generate employer descriptions once (reused across runs)
print("Generating employer descriptions via Claude API...")
descs = {}
for n, data in G.nodes(data=True):
    descs[n] = get_employer_desc(data)
print(f"Generated {len(descs)} descriptions")

goal_embedding = encoder.encode([GOAL_QUERY]).astype("float32")

all_ranked = {}
results = []
for variant_name, params in variants.items():
    with mlflow.start_run(run_name=f"{EXPERIMENT_ID}_{variant_name}"):
        mlflow.log_params(params)

        node_ids, embs = embed_nodes(G, params["include_desc"], descs)

        # Rank nodes by cosine similarity to goal query
        sims = cosine_similarity(embs, goal_embedding).flatten()
        ranked = [node_ids[i] for i in np.argsort(sims)[::-1]]
        all_ranked[variant_name] = ranked

        score = ndcg_at_k(ranked, relevant_set, k=10)
        mlflow.log_metric("ndcg_at_10", score)
        results.append({"variant": variant_name, "score": score})
        print(f"{variant_name}: NDCG@10={score:.4f}")

results_df = pd.DataFrame(results).sort_values("score", ascending=False)

# Key diagnostic: does employer_desc actually change the ranking?
corr = rank_correlation(all_ranked["no_desc"], all_ranked["with_desc"])
print(f"Rank correlation (no_desc vs with_desc): {corr:.4f}")
print("(> 0.95 means descriptions aren't changing ranking meaningfully)")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(results_df["variant"], results_df["score"])
ax.set_xlabel("NDCG@10")
ax.set_title(f"{EXPERIMENT_ID}: LLM Enrichment Impact on Ranking")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("experiments/EXP004_llm_enrichment/results.png", dpi=150)
plt.show()

print(f"\nRank correlation: {corr:.4f} (threshold 0.95)")
print(f"NDCG delta: {results_df.iloc[0]['score'] - results_df.iloc[-1]['score']:.4f} (threshold 0.02)")

## Result and Decision

**Winner:** [fill in after running]
**NDCG delta (with_desc - no_desc):** [fill in]
**Rank correlation:** [fill in — if > 0.95, descriptions add no meaningful signal]
**Decision:** [VALIDATED | INCONCLUSIVE | REJECTED]

**Decision rule:** VALIDATED only if NDCG delta ≥ 0.02 AND rank_correlation < 0.95.

**If VALIDATED — production change:**
File: `netweave/src/enrich.py`
Function: embedding input construction
Change: Include employer_desc in node text before embedding.
Notebook citation: `# Validated in EXP004 — see netweave-lab/experiments/EXP004_llm_enrichment/`